# 02 Rebuild dataset from raw BDD100K zip files

This notebook rebuilds the current project dataset directly from the three raw archives in `EcoCAR/downloads` without relying on the old notebook2 pipeline.

In [ ]:
from pathlib import Path
import json, os, sys, shutil, tarfile

ECOCAR_ROOT = Path('/content/drive/MyDrive/EcoCAR')
PROJECT_ROOT = ECOCAR_ROOT / 'DETR_GeoLane_pipeline'
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path('/content/DETR_GeoLane_pipeline')
assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DOWNLOADS = ECOCAR_ROOT / 'downloads'
RAW_ROOT = Path('/content/bdd100k_raw')
OUTPUT_ROOT = ECOCAR_ROOT / 'datasets' / 'bdd100k_vehicle5'
PACKAGE_TAR = ECOCAR_ROOT / 'datasets' / 'bdd100k_vehicle5.tar'
GLOBAL_PATHS_CONFIG = ECOCAR_ROOT / 'paths_config.yaml'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DOWNLOADS    =', DOWNLOADS)
print('RAW_ROOT     =', RAW_ROOT)
print('OUTPUT_ROOT  =', OUTPUT_ROOT)
print('PACKAGE_TAR  =', PACKAGE_TAR)
print('GLOBAL_PATHS_CONFIG =', GLOBAL_PATHS_CONFIG)


In [ ]:
from src.data_prep import inspect_download_archives

archive_preview = inspect_download_archives(DOWNLOADS)
for name, members in archive_preview.items():
    print('\n' + '='*90)
    print(name)
    if not members:
        print('NOT FOUND')
    else:
        for m in members[:30]:
            print(m)

In [ ]:
from src.data_prep import rebuild_dualpath_dataset

summary = rebuild_dualpath_dataset(
    downloads_dir=DOWNLOADS,
    raw_root=RAW_ROOT,
    output_root=OUTPUT_ROOT,
    force_reextract=False,
)

print(json.dumps(summary, indent=2))

In [ ]:
# Persist notebook02 output for notebook00
from pathlib import Path
import shutil, tarfile, os

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PACKAGE_TAR.parent.mkdir(parents=True, exist_ok=True)

# Copy dataset-local paths_config.yaml to the global location used by src.config
local_paths_cfg = OUTPUT_ROOT / 'paths_config.yaml'
if local_paths_cfg.exists():
    shutil.copy2(local_paths_cfg, GLOBAL_PATHS_CONFIG)
    print('Copied paths_config to', GLOBAL_PATHS_CONFIG)
else:
    print('WARNING: paths_config.yaml missing at', local_paths_cfg)

# Package a dereferenced tar so notebook00 can extract and use it directly.
# We add sub-items instead of the root folder so extraction produces:
#   /content/bdd100k_vehicle5/images, labels, ...
with tarfile.open(PACKAGE_TAR, 'w', dereference=True) as tar:
    for item in sorted(OUTPUT_ROOT.iterdir()):
        tar.add(item, arcname=item.name)

print('Packaged dataset tar:', PACKAGE_TAR)
print('Tar size (GB):', round(PACKAGE_TAR.stat().st_size / 1024**3, 3))
print('OUTPUT_ROOT contents:', sorted([p.name for p in OUTPUT_ROOT.iterdir()]))


In [ ]:
from collections import Counter

def scan_raw_ids(label_dir, max_files=2000):
    counter = Counter()
    txts = sorted(label_dir.glob('*.txt'))[:max_files]
    for p in txts:
        for line in p.read_text().splitlines():
            s = line.strip().split()
            if len(s) >= 5:
                counter[int(float(s[0]))] += 1
    return counter

train_counter = scan_raw_ids(OUTPUT_ROOT / 'labels' / 'train', 3000)
val_counter = scan_raw_ids(OUTPUT_ROOT / 'labels' / 'val', 1000)
print('train raw ids ->', dict(sorted(train_counter.items())))
print('val raw ids   ->', dict(sorted(val_counter.items())))
print('expected ids  -> {0: car, 1: truck, 2: bus, 3: motorcycle, 4: bicycle}')

In [ ]:
yaml_path = OUTPUT_ROOT / 'bdd100k_vehicle5.yaml'
paths_cfg = OUTPUT_ROOT / 'paths_config.yaml'
print('Dataset YAML:', yaml_path)
print(yaml_path.read_text())
print('Paths config:', paths_cfg)
print(paths_cfg.read_text())